### Data Ingestion

### document datastructure


In [63]:
from langchain_core.documents import Document

In [64]:
doc = Document(
    page_content="This is the main text content i am using to create RAG.",
    metadata={
        "source": "example.txt",
        "author": "John Doe",
        "date_created": "2024-06-01"
        }
)
doc

Document(metadata={'source': 'example.txt', 'author': 'John Doe', 'date_created': '2024-06-01'}, page_content='This is the main text content i am using to create RAG.')

### Create a txt file

In [65]:
import os
os.makedirs("../data/text_files", exist_ok=True)


In [66]:
sample_text = {
    "../data/text_files/python_intro.txt": """AI ML introduction text.
    Artificial Intelligence (AI) is a field of computer science that focuses on creating machines capable of performing tasks that typically require human intelligence. These tasks include learning from data, recognizing patterns, understanding natural language, and making decisions.

Machine Learning (ML) is a subset of AI that enables systems to learn and improve automatically without being explicitly programmed. It uses algorithms and statistical models to analyze data and make predictions.

Deep Learning is a specialized branch of Machine Learning that uses neural networks with many layers (hence “deep”) to model complex patterns in data. It is widely used in applications such as image recognition, speech processing, and natural language understanding.

Natural Language Processing (NLP) is a domain of AI that deals with the interaction between computers and human language. It enables machines to read, understand, and generate human language in a meaningful way.

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on a model’s internal knowledge, RAG retrieves relevant documents from a database and uses them to generate more accurate and up-to-date responses.

Vector databases are used in RAG systems to store embeddings, which are numerical representations of text. These embeddings help in finding similar pieces of information quickly using similarity search.

Applications of RAG include document question answering, chatbots, recommendation systems, and knowledge assistants like NotebookLM.""",
    "../data/text_files/ai_applications.txt": """Python is a high-level, interpreted programming language known for its simplicity and readability. It is widely used in various domains such as web development, data science, machine learning, automation, and software development.

One of the key features of Python is its clean and easy-to-understand syntax, which makes it an excellent choice for beginners. Python uses indentation to define code blocks, making the code visually organized and easier to read.

Python supports multiple programming paradigms, including procedural programming, object-oriented programming (OOP), and functional programming. This flexibility allows developers to choose the best approach for their problem.

Libraries are one of Python’s greatest strengths. Popular libraries include NumPy for numerical computing, Pandas for data analysis, Matplotlib for data visualization, and TensorFlow and PyTorch for machine learning and deep learning.

Python also has a large and active community, which means there are plenty of resources, tutorials, and support available for learners and developers.

In web development, frameworks like Django and Flask are commonly used to build scalable applications. In automation, Python scripts are often used to perform repetitive tasks efficiently.

Because of its versatility and ease of use, Python has become one of the most popular programming languages in the world.""",
}

for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


#### read text

In [67]:
pip install -U langchain langchain-community


Note: you may need to restart the kernel to use updated packages.


d:\Data Sciece Mastery\RAG Learnings\.venv\Scripts\python.exe: No module named pip


In [68]:

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt",encoding = "utf-8")
documents = loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='AI ML introduction text.\n    Artificial Intelligence (AI) is a field of computer science that focuses on creating machines capable of performing tasks that typically require human intelligence. These tasks include learning from data, recognizing patterns, understanding natural language, and making decisions.\n\nMachine Learning (ML) is a subset of AI that enables systems to learn and improve automatically without being explicitly programmed. It uses algorithms and statistical models to analyze data and make predictions.\n\nDeep Learning is a specialized branch of Machine Learning that uses neural networks with many layers (hence “deep”) to model complex patterns in data. It is widely used in applications such as image recognition, speech processing, and natural language understanding.\n\nNatural Language Processing (NLP) is a domain of AI that deals with the interaction between computers and human lang

In [69]:
pip install langchain langchain-community

Note: you may need to restart the kernel to use updated packages.


d:\Data Sciece Mastery\RAG Learnings\.venv\Scripts\python.exe: No module named pip


In [70]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all .txt files from 'data' folder
loader = DirectoryLoader(
    path="../data/text_files",  # directory containing text files
    glob="*.txt",          # load only .txt files
    loader_kwargs={"encoding": "utf-8"},  # specify encoding for text files
    loader_cls=TextLoader  # use TextLoader for text files
)

documents = loader.load()

# Print loaded documents
for i, doc in enumerate(documents):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:200])  # first 200 characters


--- Document 1 ---
Python is a high-level, interpreted programming language known for its simplicity and readability. It is widely used in various domains such as web development, data science, machine learning, automat

--- Document 2 ---
AI ML introduction text.
    Artificial Intelligence (AI) is a field of computer science that focuses on creating machines capable of performing tasks that typically require human intelligence. These 


In [71]:
pip install -U langchain-community pymupdf


Note: you may need to restart the kernel to use updated packages.


d:\Data Sciece Mastery\RAG Learnings\.venv\Scripts\python.exe: No module named pip


In [72]:
from langchain_community.document_loaders import PyPDFLoader
loader = DirectoryLoader(
    path="../data/pdf",  # directory containing text files
    glob="*.pdf",          # load only .txt files
    loader_cls=PyPDFLoader  # use pdf loader for pdf files
)

pdf_documents = loader.load()
pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T16:23:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T16:23:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention_2000.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Attention Mechanism\nAttention Mechanism\nThe attention mechanism is one of the most important innovations in modern artificial intelligence,\nespecially in the field of deep learning and natural language processing. It was introduced to solve a\nmajor limitation of earlier models such as recurrent neural networks (RNNs) and long short-term\nmemory networks (LSTMs), which struggled to handle long sequences effectively. These traditional\nmodels attempted to encode an entire sequence into a fixed-size vector, often losing critical\ninformation in the process. Attention provides a 

### RAG Pipelines - Data ingestion to vector dB pipelines

In [73]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

##### Data ingestion

In [74]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory (including subfolders)"""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # IMPORTANT: store documents
            all_documents.extend(documents)

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    return all_documents
docs = process_all_pdfs("../data")

print(f"\nTotal documents loaded: {len(docs)}")
print(docs[0].page_content[:200])
print(docs[0].metadata)
docs

Found 4 PDF files to process

Processing: Attention_2000.pdf

Processing: GenAI_2000.pdf

Processing: LLMs_2000.pdf

Processing: RAG_2000.pdf

Total documents loaded: 9
Attention Mechanism
Attention Mechanism
The attention mechanism is one of the most important innovations in modern artificial intelligence,
especially in the field of deep learning and natural languag
{'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T16:23:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T16:23:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention_2000.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Attention_2000.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T16:23:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T16:23:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention_2000.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Attention_2000.pdf', 'file_type': 'pdf'}, page_content='Attention Mechanism\nAttention Mechanism\nThe attention mechanism is one of the most important innovations in modern artificial intelligence,\nespecially in the field of deep learning and natural language processing. It was introduced to solve a\nmajor limitation of earlier models such as recurrent neural networks (RNNs) and long short-term\nmemory networks (LSTMs), which struggled to handle long sequences effectively. These traditional\nmodels attempted to encode an entire sequence into a fixed-size vector, often losing cr

### Chunking

In [ ]:
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_docs(docs):
    seen = set()
    cleaned = []

    for doc in docs:
        text = " ".join(doc.page_content.split())  # normalize whitespace
        if text not in seen:
            seen.add(text)
            doc.page_content = text
            cleaned.append(doc)

    return cleaned


def split_documents(documents, chunk_size=400, chunk_overlap=80):
    """Split documents into smaller deduplicated chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    docs = splitter.split_documents(documents)
    docs = clean_docs(docs)

    print(f"Split {len(documents)} documents into {len(docs)} unique chunks")

    if docs:
        print(f"Example chunk content: {docs[0].page_content[:200]}")
        print(f"Example chunk metadata: {docs[0].metadata}")
    return docs


In [76]:
chunks = split_documents(docs)
chunks

Split 9 documents into 34 chunks
Example chunk content: Attention Mechanism
Attention Mechanism
The attention mechanism is one of the most important innovations in modern artificial intelligence,
especially in the field of deep learning and natural languag
Example chunk metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T16:23:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T16:23:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention_2000.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Attention_2000.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T16:23:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T16:23:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention_2000.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Attention_2000.pdf', 'file_type': 'pdf'}, page_content='Attention Mechanism\nAttention Mechanism\nThe attention mechanism is one of the most important innovations in modern artificial intelligence,\nespecially in the field of deep learning and natural language processing. It was introduced to solve a\nmajor limitation of earlier models such as recurrent neural networks (RNNs) and long short-term\nmemory networks (LSTMs), which struggled to handle long sequences effectively. These traditional\nmodels attempted to encode an entire sequence into a fixed-size vector, often losing cr

### Embedding and vector store DB

In [77]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


d:\Data Sciece Mastery\RAG Learnings\.venv\Scripts\python.exe: No module named pip


In [78]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

In [79]:
class EmbeddingManager:
    """Handle document embedding generation and sentence transformer model loading"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager 
        Args:
            model_name: huggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name) # this were the the system starts understanding the model loading process. Without this its just a text storage.
            print(f"Model loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
    # Step 1: Tell user model is loading
    # Step 2: Load model into memory
    # Step 3: Confirm success + show vector size
    # Step 4: If fail → show error + stop program

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts: list of strings to embed
        Returns:
            np.ndarray: array of embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    ## initalize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1688.24it/s]


Model loaded successfully.Embedding dimension: 384


### Vecor Store

In [80]:
class VectorStore:
    """Simple vector store using ChromaDB for storing and retrieving document embeddings"""
    def __init__(self, collection_name: str = "documents",persist_directory: str = "../data/vector_store"):
        """Initialize the vector store
        Args:
            collection_name: name of the ChromaDB collection to use
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}...")
            
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Pdf document for RAG system"}
            )
            print(f"VectorStore initialized with collection name: {self.collection_name}")
            print(f"existing document in collection: {self.collection.count()}") # this is the line where we are checking if the collection already has documents or not. If it does, we can decide to clear it or not based on our needs.

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e
        

# 1. Create folder (if needed)
# 2. Connect to ChromaDB
# 3. Create/load collection
# 4. Check existing data
# 5. Ready to store embeddings

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents to the vector store with embeddings
        Args:
            documents: list of Document objects to add
            embedding_manager: instance of EmbeddingManager to generate embeddings
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store..." )

        #prepare the data for adding to chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embeddings) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" # unique id for each document
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            documents_text.append(doc.page_content)

            #embeddings
            embeddings_list.append(embeddings)
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Documents added successfully {len(documents)} documents added.")
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise e

VectorStore = VectorStore()
VectorStore


Initializing ChromaDB client with persist directory: ../data/vector_store...
VectorStore initialized with collection name: documents
existing document in collection: 42


In [81]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T16:23:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T16:23:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention_2000.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Attention_2000.pdf', 'file_type': 'pdf'}, page_content='Attention Mechanism\nAttention Mechanism\nThe attention mechanism is one of the most important innovations in modern artificial intelligence,\nespecially in the field of deep learning and natural language processing. It was introduced to solve a\nmajor limitation of earlier models such as recurrent neural networks (RNNs) and long short-term\nmemory networks (LSTMs), which struggled to handle long sequences effectively. These traditional\nmodels attempted to encode an entire sequence into a fixed-size vector, often losing cr

In [82]:
# Conver the text to embeddings

text= [doc.page_content for doc in chunks]
text

['Attention Mechanism\nAttention Mechanism\nThe attention mechanism is one of the most important innovations in modern artificial intelligence,\nespecially in the field of deep learning and natural language processing. It was introduced to solve a\nmajor limitation of earlier models such as recurrent neural networks (RNNs) and long short-term\nmemory networks (LSTMs), which struggled to handle long sequences effectively. These traditional\nmodels attempted to encode an entire sequence into a fixed-size vector, often losing critical\ninformation in the process. Attention provides a solution by allowing the model to dynamically focus\non different parts of the input sequence during processing.\nAt its core, attention works by computing relationships between elements of a sequence. It involves\nthree main components: queries, keys, and values. A query is compared with a set of keys to\nproduce similarity scores, which are then normalized using a softmax function. These scores are',
 'thre

In [83]:
# Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(text)

# store in the vector database
VectorStore.add_documents(chunks, embeddings)


Generating embeddings for 34 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.36s/it]


generated embeddings with shape: (34, 384)
Adding 34 documents to vector store...
Documents added successfully 34 documents added.
Documents added successfully. Total documents in collection: 76


### Retrival Pipeline from vectorstore

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

class RAGretriever:
    """Retriever for RAG system to fetch relevant documents based on query"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager, top_k: int = 6):
        """Initialize the retriever
        Args:
            vector_store: instance of VectorStore to retrieve from
            embedding_manager: instance of EmbeddingManager to generate query embeddings
            top_k: number of top relevant documents to retrieve
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.keyword_docs = self._load_keyword_docs()
        self.bm25 = self._build_bm25(self.keyword_docs)
        self.reranker = None

    def _get_text(self, doc):
        if isinstance(doc, dict):
            return doc.get("document", "")
        return getattr(doc, "page_content", str(doc))

    def _get_metadata(self, doc):
        if isinstance(doc, dict):
            return doc.get("metadata") or {}
        return getattr(doc, "metadata", {}) or {}

    def _get_page_number(self, doc):
        page = self._get_metadata(doc).get("page", 0)
        try:
            return int(page)
        except (TypeError, ValueError):
            return 0

    def _clean_retrieved_docs(self, docs):
        seen_text = set()
        seen_meta = set()
        cleaned = []

        for doc in docs:
            text = " ".join(self._get_text(doc).split())
            metadata = self._get_metadata(doc)
            meta_key = (metadata.get("source") or metadata.get("source_file"), metadata.get("page"))
            if text not in seen_text and meta_key not in seen_meta:
                seen_text.add(text)
                seen_meta.add(meta_key)
                if isinstance(doc, dict):
                    doc["document"] = text
                else:
                    doc.page_content = text
                cleaned.append(doc)

        return cleaned

    def _load_keyword_docs(self):
        stored = self.vector_store.collection.get(include=["documents", "metadatas"])
        docs = []
        for rank, (doc_id, document, metadata) in enumerate(
            zip(stored.get("ids", []), stored.get("documents", []), stored.get("metadatas", [])),
            start=1,
        ):
            docs.append({
                "id": doc_id,
                "document": document,
                "metadata": dict(metadata or {}),
                "similarity_score": None,
                "distance": None,
                "rank": rank,
            })
        return self._clean_retrieved_docs(docs)

    def _build_bm25(self, docs):
        corpus = [self._get_text(doc) for doc in docs]
        tokenized_corpus = [doc.split() for doc in corpus]
        return BM25Okapi(tokenized_corpus) if tokenized_corpus else None

    def bm25_search(self, query, docs=None, k=6):
        docs = docs or self.keyword_docs
        if not docs or self.bm25 is None:
            return []

        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [doc.copy() if isinstance(doc, dict) else doc for _, doc in ranked[:k]]

    def _get_reranker(self):
        if self.reranker is None:
            self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        return self.reranker

    def _rerank(self, query, docs, k=6):
        if not docs:
            return []

        try:
            pairs = [(query, self._get_text(doc)) for doc in docs]
            scores = self._get_reranker().predict(pairs)
            ranked_docs = [doc for _, doc in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)]
            return sorted(ranked_docs[:k], key=self._get_page_number)
        except Exception as exc:
            print(f"Reranker unavailable, using hybrid retrieval order: {exc}")
            return sorted(docs[:k], key=self._get_page_number)
    
    def _mmr_select(self, query_embedding: np.ndarray, candidate_embeddings: List[List[float]], k: int, lambda_mult: float = 0.5) -> List[int]:
        if candidate_embeddings is None or len(candidate_embeddings) == 0:
            return []

        candidates = np.asarray(candidate_embeddings, dtype=np.float32)
        query_vector = np.asarray(query_embedding, dtype=np.float32).reshape(-1)

        candidate_norms = np.linalg.norm(candidates, axis=1, keepdims=True)
        query_norm = np.linalg.norm(query_vector)
        candidate_norms[candidate_norms == 0] = 1.0
        if query_norm == 0:
            query_norm = 1.0

        normalized_candidates = candidates / candidate_norms
        normalized_query = query_vector / query_norm
        query_scores = normalized_candidates @ normalized_query

        selected = []
        remaining = list(range(len(candidates)))
        while remaining and len(selected) < k:
            if not selected:
                next_idx = max(remaining, key=lambda idx: query_scores[idx])
            else:
                selected_vectors = normalized_candidates[selected]
                next_idx = max(
                    remaining,
                    key=lambda idx: lambda_mult * query_scores[idx] - (1 - lambda_mult) * np.max(selected_vectors @ normalized_candidates[idx]),
                )
            selected.append(next_idx)
            remaining.remove(next_idx)
        return selected

    def _vector_search(self, query:str, top_k: int = 6, fetch_k: int = 20) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=max(fetch_k, top_k),
            include=["documents", "metadatas", "distances", "embeddings"],
        )

        retrive_docs = []
        documents = results.get('documents', [[]])[0]
        metadatas = results.get('metadatas', [[]])[0]
        distances = results.get('distances', [[]])[0]
        ids = results.get('ids', [[]])[0]
        candidate_embeddings = results.get('embeddings', [[]])[0]
        selected_indexes = self._mmr_select(query_embedding, candidate_embeddings, top_k, lambda_mult=0.5) or list(range(min(top_k, len(documents))))

        for rank, index in enumerate(selected_indexes, start=1):
            distance = distances[index]
            similarity_score = 1 / (1 + distance)
            metadata = dict(metadatas[index] or {})
            metadata["score"] = similarity_score
            retrive_docs.append({
                "id": ids[index],
                "document": documents[index],
                "metadata": metadata,
                "similarity_score": similarity_score,
                'distance': distance,
                'rank': rank
            })
        return retrive_docs

    def retrieve(self, query:str, top_k: int = 6, score_threshold: float | None = None) -> List[Dict[str, Any]]:
        """Retrieve relevant documents with hybrid vector + BM25 retrieval and CrossEncoder reranking."""
        print(f"Retrieving documents for query: '{query}'")
        try:
            vector_docs = self._vector_search(query, top_k=top_k, fetch_k=20)
            keyword_docs = self.bm25_search(query, self.keyword_docs, k=6)
            combined_docs = self._clean_retrieved_docs(vector_docs + keyword_docs)

            if len(combined_docs) == 0:
                combined_docs = self._clean_retrieved_docs(self._vector_search(query, top_k=top_k, fetch_k=20))

            docs = self._rerank(query, combined_docs, k=top_k)
            docs = sorted(self._clean_retrieved_docs(docs), key=self._get_page_number)[:6]

            if docs:
                print(f"Retrieved {len(docs)} reranked unique documents from hybrid search.")
            else:
                print("No relevant documents found in vector store.")
            return docs
        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise
        
rag_retriever = RAGretriever(VectorStore, embedding_manager)


       

In [85]:
rag_retriever

In [86]:
rag_retriever.retrieve("What are the main techniques used in generative AI?")

Retrieving documents for query: 'What are the main techniques used in generative AI?'
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 30.73it/s]

generated embeddings with shape: (1, 384)
Retrieved 5 relevant documents from vector store.


[{'id': 'doc_dced9886_14',
  'document': 'represents a shift from traditional AI systems to more creative and interactive systems.Generative\nAI refers to artificial intelligence systems that can create new content such as text, images, audio,\nand video. These systems are designed to mimic human creativity by learning patterns from large',
  'metadata': {'creationdate': '2026-05-01T16:23:24+00:00',
   'page_label': '1',
   'total_pages': 2,
   'content_length': 296,
   'trapped': '/False',
   'creator': '(unspecified)',
   'keywords': '',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\GenAI_2000.pdf',
   'title': '(anonymous)',
   'moddate': '2026-05-01T16:23:24+00:00',
   'author': '(anonymous)',
   'doc_index': 14,
   'source_file': 'GenAI_2000.pdf',
   'subject': '(unspecified)',
   'page': 0,
   'producer': 'ReportLab PDF Library - www.reportlab.com'},
  'similarity_score': 0.6224166073764081,
  'distance': 0.6066409349441528,
  'rank': 1},
 {'id': 'doc_6526e57b_15',
  'docum

In [87]:
rag_retriever.retrieve("What are the main techniques used in generative AI?")


Retrieving documents for query: 'What are the main techniques used in generative AI?'
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.20it/s]

generated embeddings with shape: (1, 384)
Retrieved 5 relevant documents from vector store.


[{'id': 'doc_dced9886_14',
  'document': 'represents a shift from traditional AI systems to more creative and interactive systems.Generative\nAI refers to artificial intelligence systems that can create new content such as text, images, audio,\nand video. These systems are designed to mimic human creativity by learning patterns from large',
  'metadata': {'author': '(anonymous)',
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'doc_index': 14,
   'source': '..\\data\\pdf\\GenAI_2000.pdf',
   'moddate': '2026-05-01T16:23:24+00:00',
   'title': '(anonymous)',
   'page_label': '1',
   'trapped': '/False',
   'creator': '(unspecified)',
   'keywords': '',
   'source_file': 'GenAI_2000.pdf',
   'creationdate': '2026-05-01T16:23:24+00:00',
   'content_length': 296,
   'total_pages': 2,
   'subject': '(unspecified)',
   'page': 0,
   'file_type': 'pdf'},
  'similarity_score': 0.6224166073764081,
  'distance': 0.6066409349441528,
  'rank': 1},
 {'id': 'doc_6526e57b_15',
  'docum